# Notebook 3 - Sentiment Analysis

## 0. Importing modules and downloading Dataset

Setup all the imports.

In [1]:
import pandas as pd
import numpy as np
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

Instantiate the variables with the dataset name and the paths.

In [2]:
dataset = "ashukr/rnnsentiment-data"
path = f"../data/{dataset}"
labels_path = path + "/labels.txt"
reviews_path = path + "/reviews.txt"

Download and load the dataset.

In [3]:
print(f"Downloading dataset '{dataset}' and saving it on '{path}'...")

kagglehub.dataset_download(dataset, output_dir=path)

print("Dataset downloaded!")

Dataset downloaded!


In [4]:
print(f"Loading dataset '{dataset}' and saving it on '{labels_path}' and '{reviews_path}'...")

data = pd.read_csv(reviews_path, header=None)
labels = pd.read_csv(labels_path, header=None)
Y = (labels=='positive').astype(np.int_)

data = data.values.tolist()
data= [x[0] for x in data]

labels = labels.values.tolist()
labels = [x[0] for x in labels]

print("Dataset loaded!")

Loading dataset 'ashukr/rnnsentiment-data' and saving it on '../data/ashukr/rnnsentiment-data/labels.txt' and '../data/ashukr/rnnsentiment-data/reviews.txt'...
Dataset loaded!


## 1. Preparing data

In [5]:
data_train, data_test, labels_train, labels_test = train_test_split(data, Y, test_size=0.2)

## 2. Training a neural network

Now we use the `CountVectorizer` from `sklearn.feature_extraction.text` to create a Bag-of-Words representation of the reviews.

In [6]:
vectorizer = CountVectorizer(max_features=10000)
vectorizer = vectorizer.fit(data_train)
bag_of_words_train = vectorizer.transform(data_train)
bag_of_words_test = vectorizer.transform(data_test)

Train a neural network with a single hidden layer on the dataset, tuning the relevant hyperparameters to optimize accuracy.

In [11]:
def train(net, train_values, train_labels, test_values, test_labels, num_epochs):
    x_train = torch.FloatTensor(train_values)
    x_test  = torch.FloatTensor(test_values)
    y_train = torch.FloatTensor(train_labels.values).view(-1, 1)
    y_test  = torch.FloatTensor(test_labels.values).view(-1, 1)

    train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=32, shuffle=True)
    test_loader  = DataLoader(TensorDataset(x_test,  y_test),  batch_size=32)

    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    net.to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=1e-4)

    for epoch in range(num_epochs):
        net.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            loss = criterion(net(X_batch), y_batch)
            loss.backward()
            optimizer.step()

        net.eval()
        correct = total = 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                preds = (net(X_batch) > 0.5).float()   # threshold instead of argmax
                correct += (preds == y_batch).sum().item()
                total   += y_batch.size(0)

        print(f"Epoch {epoch+1:>3} | Test acc: {correct/total:.4f}")
    return net

def evaluate(net, X, y, criterion=None):
    device = next(net.parameters()).device
    if criterion is None:
        criterion = nn.BCELoss()

    net.eval()
    with torch.no_grad():
        X_t = torch.FloatTensor(np.asarray(X)).to(device)
        y_t = torch.FloatTensor(np.asarray(y).reshape(-1)).view(-1, 1).to(device)

        preds = net(X_t)
        loss = criterion(preds, y_t).item()

        predicted_labels = (preds > 0.5).float()
        accuracy = (predicted_labels == y_t).float().mean().item()

    return loss, accuracy

In [8]:
class Net(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        return x

In [9]:
net = Net(10000)

X = bag_of_words_train.toarray()
y = labels_train

val_size = int(len(X) * 0.2)
X_train, X_val = X[:-val_size], X[-val_size:]
y_train, y_val = y[:-val_size], y[-val_size:]

net = train(net, X_train, y_train, X_val, y_val, 2)

/tmp/ipykernel_33183/4162582598.py:4: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  y_train = torch.FloatTensor(train_labels.values).view(-1, 1)


Epoch   1 | Test acc: 0.8350
Epoch   2 | Test acc: 0.8600


Test your sentiment-classifier on the test set.

In [12]:
train_loss, train_acc = evaluate(net, bag_of_words_train.toarray(), labels_train)
print(f"Loss + accuracy on train data: [{train_loss:.4f}, {train_acc:.4f}]")

test_loss, test_acc = evaluate(net, bag_of_words_test.toarray(), labels_test)
print(f"Loss + accuracy on test data: [{test_loss:.4f}, {test_acc:.4f}]")

Loss + accuracy on train data: [0.3676, 0.8887]
Loss + accuracy on test data: [0.3863, 0.8782]
